# Lab 04: Portal, SDK & Custom Evaluations

## 🎯 Objectives

Execute evaluations through both Foundry Portal and SDK, then build custom evaluators:
1. Run SDK evaluation with built-in evaluators
2. Build a **code-based** custom evaluator (deterministic)
3. Build a **prompt-based** custom evaluator (LLM-as-judge)
4. Run all eight evaluators against the **same existing agent** and inspect results in Foundry
5. Export actual scores, evaluator versions, run IDs, and the portal report URL for Lab 7

## ⏱️ Duration: 60-75 minutes

## 🔐 Authentication

```bash
az login --use-device-code
```

First complete [Lab 03B: reference-agent setup](../lab-03-evaluator-matrix/lab-03b-reference-agent-setup.ipynb) and save the resolved agent name/version in `.env`. Use the existing project configured there. The default agent is `contoso-support-agent-eval`; `AI_FOUNDRY_AGENT_NAME` and `AI_FOUNDRY_AGENT_VERSION` select the pinned target. This notebook never creates or updates an agent.

Custom evaluator catalog APIs are in preview. You need permission to read the agent, create evaluator versions and evaluation runs, and invoke the configured judge deployment. Running the cloud evaluation incurs model usage.

In [ ]:
import os
import json
import time
from pathlib import Path
from datetime import datetime, timezone
from dotenv import load_dotenv

env_path = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / '.env.template').is_file()) / '.env'
load_dotenv(env_path)
repo_root = env_path.parent
lab_dir = repo_root / "labs" / "lab-04-portal-sdk-custom"

project_endpoint = os.environ.get("AI_FOUNDRY_PROJECT_ENDPOINT")
tenant_id = os.environ.get("TENANT_ID")
model_deployment = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")
agent_name = os.environ.get("AI_FOUNDRY_AGENT_NAME", "contoso-support-agent-eval")
agent_version = os.environ.get("AI_FOUNDRY_AGENT_VERSION")

if not project_endpoint:
    raise ValueError("🚨 AI_FOUNDRY_PROJECT_ENDPOINT not set in .env")

print(f"📍 Project: {project_endpoint[:50]}...")
print(f"🤖 Model: {model_deployment}")

In [ ]:
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.core.exceptions import ResourceNotFoundError
from openai.types.eval_create_params import DataSourceConfigCustom

credential = AzureCliCredential(tenant_id=tenant_id)
preview_headers = {"Foundry-Features": "Evaluations=V1Preview"}
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential, headers=preview_headers)
openai_client = project_client.get_openai_client(default_headers=preview_headers)
# The evaluator catalog moved out of beta in newer SDK versions.
evaluator_catalog = project_client.evaluators if hasattr(project_client, "evaluators") else project_client.beta.evaluators

print("✅ Clients initialized")

## Part 1: Reuse the Existing Agent

Resolve the requested version once and pin it for this run. Leave the agent's instructions, tools, and deployment unchanged.

In [ ]:
def resolve_agent(project_client, name, version=None):
    try:
        if version:
            return project_client.agents.get_version(agent_name=name, agent_version=version)
        return project_client.agents.get(agent_name=name).versions.latest
    except ResourceNotFoundError as exc:
        raise RuntimeError(f"Existing agent {name!r} (version {version or 'latest'}) was not found. Check AI_FOUNDRY_AGENT_NAME / AI_FOUNDRY_AGENT_VERSION; no agent was created.") from exc

agent = resolve_agent(project_client, agent_name, agent_version)
print(f"Reusing agent: {agent.name} (version: {agent.version})")

## Part 2: Prepare the Built-in Evaluators and Agent Target

Prepare quality + safety criteria and the golden dataset. We submit **one combined cloud run** in Part 5, after adding the custom evaluators.

In [ ]:
# Configure data source schema
data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "ground_truth": {"type": "string"},
            "context": {"type": "string"},
            "category": {"type": "string"}
        },
        "required": ["query"]
    },
    include_sample_schema=True,
)

# Define comprehensive testing criteria
testing_criteria = [
    # Quality evaluators
    {
        "type": "azure_ai_evaluator",
        "name": "Task Adherence",
        "evaluator_name": "builtin.task_adherence",
        "initialization_parameters": {"deployment_name": model_deployment},
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_items}}"
        },
    },
    {
        "type": "azure_ai_evaluator",
        "name": "Coherence",
        "evaluator_name": "builtin.coherence",
        "initialization_parameters": {"deployment_name": model_deployment},
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}"
        },
    },
    {
        "type": "azure_ai_evaluator",
        "name": "Fluency",
        "evaluator_name": "builtin.fluency",
        "initialization_parameters": {"deployment_name": model_deployment},
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}"
        },
    },
    # Safety evaluators
    {
        "type": "azure_ai_evaluator",
        "name": "Violence",
        "evaluator_name": "builtin.violence",
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}"
        },
    },
    {
        "type": "azure_ai_evaluator",
        "name": "Sexual Content",
        "evaluator_name": "builtin.sexual",
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}"
        },
    },
    {
        "type": "azure_ai_evaluator",
        "name": "Hate/Unfairness",
        "evaluator_name": "builtin.hate_unfairness",
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{sample.output_text}}"
        },
    },
]

print(f"✅ Testing criteria configured: {len(testing_criteria)} evaluators")
for tc in testing_criteria:
    print(f"   • {tc['name']} ({tc['evaluator_name']})")

In [ ]:
builtin_criteria = list(testing_criteria)
print(f"Prepared {len(builtin_criteria)} built-in evaluators; custom criteria follow in Part 5.")

In [ ]:
# Load test queries from golden dataset
dataset_path = repo_root / "datasets" / "golden" / "contoso-golden-v1.jsonl"

test_queries = []
with dataset_path.open(encoding="utf-8") as f:
    for line in f:
        entry = json.loads(line)
        test_queries.append({"item": {
            "query": entry["query"],
            "ground_truth": entry.get("ground_truth", ""),
            "context": entry.get("context", ""),
            "category": entry["category"],
        }})

print(f"📝 Loaded {len(test_queries)} test queries from golden dataset")

In [ ]:
# Configure data source targeting the agent
data_source = {
    "type": "azure_ai_target_completions",
    "source": {
        "type": "file_content",
        "content": test_queries,
    },
    "input_messages": {
        "type": "template",
        "template": [
            {
                "type": "message",
                "role": "user",
                "content": {"type": "input_text", "text": "{{item.query}}"}
            }
        ],
    },
    "target": {
        "type": "azure_ai_agent",
        "name": agent.name,
        "version": agent.version,
    },
}

print(f"Cloud target prepared: {agent.name}, version {agent.version}")

In [ ]:
print(f"Ready to evaluate {len(test_queries)} golden queries. Ground truth is reference data, NOT a simulated agent response.")

## Part 3: Custom Code-Based Evaluator

Code-based evaluators are **deterministic** — they check format, schema, length, keywords, or business rules without needing an LLM judge. The following local tests explain the rules; Part 5 registers the same logic in Foundry.

In [ ]:
# Custom code-based evaluator: Response Quality Checker
def response_quality_checker(response: str, query: str) -> dict:
    """
    Deterministic evaluator that checks:
    1. Response is not empty
    2. Response length is reasonable (50-2000 chars)
    3. Response doesn't contain forbidden phrases
    4. Response includes disclaimer for sensitive topics
    """
    issues = []
    score = 5  # Start with perfect score
    
    # Check 1: Not empty
    if not response or len(response.strip()) == 0:
        return {"score": 1, "label": "fail", "reasoning": "Empty response"}
    
    # Check 2: Length bounds
    if len(response) < 50:
        score -= 1
        issues.append("Response too short (<50 chars)")
    elif len(response) > 2000:
        score -= 1
        issues.append("Response too long (>2000 chars)")
    
    # Check 3: Forbidden phrases
    forbidden = ["as an AI", "I cannot help", "I'm just a language model"]
    for phrase in forbidden:
        if phrase.lower() in response.lower():
            score -= 1
            issues.append(f"Contains forbidden phrase: '{phrase}'")
    
    # Check 4: Sensitive topic disclaimer
    sensitive_keywords = ["salary", "compensation", "termination", "disciplinary"]
    is_sensitive = any(kw in query.lower() for kw in sensitive_keywords)
    if is_sensitive:
        has_disclaimer = any(d in response.lower() for d in ["consult", "contact hr", "hr business partner", "confidential"])
        if not has_disclaimer:
            score -= 1
            issues.append("Sensitive topic without appropriate disclaimer/redirect")
    
    score = max(1, score)  # Floor at 1
    label = "pass" if score >= 4 else "needs_review" if score >= 3 else "fail"
    
    return {
        "score": score,
        "label": label,
        "reasoning": "; ".join(issues) if issues else "All checks passed"
    }

# Test the evaluator
test_cases = [
    ("What is annual leave?", "Full-time employees receive 20 days of annual leave per year, accruing monthly."),
    ("What is annual leave?", "Yes."),  # Too short
    ("Tell me about salaries", "The salary ranges are between 50k and 150k depending on level."),  # Missing disclaimer
]

print("🧪 Code-Based Evaluator Test Results:")
for query, response in test_cases:
    result = response_quality_checker(response, query)
    print(f"   Q: {query[:40]}...")
    print(f"   R: {response[:40]}...")
    print(f"   → Score: {result['score']}/5, Label: {result['label']}, Reason: {result['reasoning']}")
    print()

## Part 4: Custom Prompt-Based Evaluator (LLM-as-Judge)

Prompt-based evaluators use an LLM judge with a custom rubric. Best for **subjective** quality assessment. These sample judgments are local demonstrations; the registered evaluator's scores and reasons appear in the cloud run in Part 5.

In [ ]:
# LLM-as-Judge: Professional Tone Evaluator
professional_tone_prompt = """
You are evaluating the professional tone of an enterprise support agent's response.

## Scoring Rubric

Score the response on a 1-5 scale:

**5 - Excellent**: Professional, empathetic, clear, appropriate formality for workplace. 
Uses active voice. Provides actionable next steps.

**4 - Good**: Professional and clear. Minor tone issues (slightly too casual or formal). 
Still appropriate for workplace communication.

**3 - Acceptable**: Functional but could be improved. May be overly robotic, 
too verbose, or slightly condescending. Still usable.

**2 - Below Standard**: Unprofessional elements present. Too casual, uses slang, 
dismissive, or overly technical without explanation.

**1 - Unacceptable**: Rude, offensive, inappropriate for workplace, or completely 
fails to address the user respectfully.

## Input
- User Query: {query}
- Agent Response: {response}

## Output Format
Respond with ONLY a JSON object:
{{
    "score": <1-5>,
    "reasoning": "<brief explanation of score>"
}}
"""

print("✅ Professional Tone Evaluator prompt defined")
print(f"   Prompt length: {len(professional_tone_prompt)} chars")

In [ ]:
# Run LLM-as-judge evaluation
def evaluate_professional_tone(query: str, response: str) -> dict:
    """Use the configured deployment as the professional-tone judge."""
    prompt = professional_tone_prompt.format(query=query, response=response)
    
    judge_response = openai_client.chat.completions.create(
        model=model_deployment,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,  # Reduce variation; an LLM judge is not deterministic.
        response_format={"type": "json_object"}
    )
    
    result = json.loads(judge_response.choices[0].message.content)
    result["label"] = "pass" if result["score"] >= 4 else "needs_review" if result["score"] >= 3 else "fail"
    result["tokens_used"] = judge_response.usage.total_tokens
    return result

# Test with sample responses
judge_test_cases = [
    (
        "How do I request time off?",
        "Great question! To request time off, navigate to the HR Portal and select 'Leave Request'. Submit your dates at least 2 weeks in advance for periods over 3 days. Your manager will receive a notification for approval. Let me know if you need any other help!"
    ),
    (
        "How do I request time off?",
        "idk just go to the portal thing and click around lol. its somewhere in there. or ask your boss i guess"
    ),
]

print("🧪 LLM-as-Judge Test Results:")
for query, response in judge_test_cases:
    result = evaluate_professional_tone(query, response)
    print(f"   Q: {query}")
    print(f"   R: {response[:60]}...")
    print(f"   → Score: {result['score']}/5, Label: {result['label']}")
    print(f"   → Reasoning: {result['reasoning']}")
    print(f"   → Tokens used: {result['tokens_used']}")
    print()

## Part 5: Run Full Evaluation Combining All Evaluators

Combine built-in + code-based + prompt-based evaluators into a single **cloud** evaluation run.

The code evaluator reuses the Python function above, adapting its 1-5 score to Foundry's 0-1 contract: `(score - 1) / 4`. The pass threshold is 0.75 (equivalent to 4/5). For agent targets, generated text is available at `item['sample']['output_text']`; it is never substituted with ground truth.

The prompt evaluator reuses the tone rubric with Foundry's `{{query}}` / `{{response}}` placeholders and `result` / `reason` output contract. Its pass threshold is 4/5.

Existing evaluator definitions, input schemas, initialization parameters, and metric ranges must match before reuse. An incompatible existing name raises an error rather than silently changing the rubric or overwriting it. Optional `.env` evaluator names/versions let you select compatible catalog entries. Missing default entries are created once; explicitly requested missing versions are errors.

[Custom evaluator SDK contracts (preview)](https://learn.microsoft.com/azure/foundry/concepts/evaluation-evaluators/custom-evaluators)

In [ ]:
import inspect
import textwrap

code_name = os.environ.get("LAB04_CODE_EVALUATOR_NAME", "contoso_response_quality_checker")
tone_name = os.environ.get("LAB04_TONE_EVALUATOR_NAME", "contoso_professional_tone")
code_text = textwrap.dedent(inspect.getsource(response_quality_checker)) + '''

def grade(sample: dict, item: dict) -> float:
    response = item['sample']['output_text']
    query = item['query']
    if not isinstance(response, str) or not isinstance(query, str):
        raise TypeError('Expected string query and generated response')
    return (response_quality_checker(response, query)['score'] - 1) / 4.0
'''
tone_prompt = professional_tone_prompt.format(query="{{query}}", response="{{response}}")
tone_prompt = tone_prompt.replace('"score":', '"result":').replace('"reasoning":', '"reason":')

def evaluator_spec(name, display_name, definition):
    return {"name": name, "display_name": display_name, "evaluator_type": "custom",
            "categories": ["quality"], "definition": definition}

code_spec = evaluator_spec(code_name, "Response Quality Checker", {
    "type": "code", "code_text": code_text,
    "init_parameters": {
        "type": "object", "properties": {
            "deployment_name": {"type": "string"}, "pass_threshold": {"type": "number"}},
        "required": ["deployment_name", "pass_threshold"]},
    "metrics": {"result": {"type": "continuous", "desirable_direction": "increase", "min_value": 0.0, "max_value": 1.0}},
    "data_schema": {
        "type": "object", "required": ["item"], "properties": {
            "item": {"type": "object", "required": ["query", "sample"], "properties": {
                "query": {"type": "string"},
                "sample": {"type": "object", "required": ["output_text"], "properties": {
                    "output_text": {"type": "string"}}}}}}},
})
tone_spec = evaluator_spec(tone_name, "Professional Tone", {
    "type": "prompt", "prompt_text": tone_prompt,
    "init_parameters": {
        "type": "object", "properties": {
            "deployment_name": {"type": "string"}, "threshold": {"type": "number"}},
        "required": ["deployment_name", "threshold"]},
    "metrics": {"professional_tone": {"type": "ordinal", "desirable_direction": "increase", "min_value": 1, "max_value": 5}},
    "data_schema": {"type": "object", "required": ["query", "response"], "properties": {
        "query": {"type": "string"}, "response": {"type": "string"}}},
})

def reuse_or_register_evaluator(catalog, spec, version=None):
    name = spec["name"]
    if version:
        existing = catalog.get_version(name=name, version=version)
    else:
        try:
            versions = list(catalog.list_versions(name=name))
        except ResourceNotFoundError:
            versions = []
        if not versions:
            created = catalog.create_version(name=name, evaluator_version=spec)
            print(f"Registered evaluator: {name}, version {created.version}")
            return created
        existing = max(versions, key=lambda entry: int(entry.version))
    definition = existing.definition.as_dict()
    mismatches = [key for key, value in spec["definition"].items() if definition.get(key) != value]
    if existing.evaluator_type != "custom" or mismatches:
        raise ValueError(f"Evaluator {name!r}, version {existing.version}, is incompatible: {mismatches or ['evaluator_type']}. Review its definition in Foundry, select a compatible version or use a new LAB04 evaluator name. Nothing was overwritten.")
    print(f"Reusing evaluator: {name}, version {existing.version} (definition and input contract verified)")
    return existing

print("Existing custom evaluator catalog:")
for entry in evaluator_catalog.list(type="custom"):
    print(f"  {entry.name}, version {entry.version}, type {entry.definition.type}")
code_evaluator = reuse_or_register_evaluator(evaluator_catalog, code_spec, os.environ.get("LAB04_CODE_EVALUATOR_VERSION"))
tone_evaluator = reuse_or_register_evaluator(evaluator_catalog, tone_spec, os.environ.get("LAB04_TONE_EVALUATOR_VERSION"))

custom_criteria = [
    {"type": "azure_ai_evaluator", "name": "Response Quality Checker",
     "evaluator_name": code_evaluator.name, "evaluator_version": code_evaluator.version,
     "initialization_parameters": {"deployment_name": model_deployment, "pass_threshold": 0.75}},
    {"type": "azure_ai_evaluator", "name": "Professional Tone",
     "evaluator_name": tone_evaluator.name, "evaluator_version": tone_evaluator.version,
     "initialization_parameters": {"deployment_name": model_deployment, "threshold": 4},
     "data_mapping": {"query": "{{item.query}}", "response": "{{sample.output_text}}"}},
]
testing_criteria = builtin_criteria + custom_criteria
print(f"Ready: {len(testing_criteria)} evaluators against {agent.name}, version {agent.version}")

In [ ]:
def wait_for_run(client, eval_id, run, timeout_seconds=1800, poll_seconds=10):
    deadline = time.monotonic() + timeout_seconds
    while run.status not in {"completed", "failed", "canceled", "cancelled"}:
        if time.monotonic() >= deadline:
            raise TimeoutError(f"Run {run.id} is still {run.status}. It has NOT been canceled. Inspect it in Foundry or retrieve it by eval ID {eval_id}; do not submit a duplicate run.")
        time.sleep(poll_seconds)
        run = client.evals.runs.retrieve(eval_id=eval_id, run_id=run.id)
        print(f"Status: {run.status}")
    if run.status != "completed":
        raise RuntimeError(f"Evaluation {eval_id}, run {run.id}: {run.status}; error: {getattr(run, 'error', None)}")
    return run

eval_object = openai_client.evals.create(
    name="Contoso Support Agent - Built-in and Custom Baseline",
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,
)
eval_run = openai_client.evals.runs.create(
    eval_id=eval_object.id,
    name=f"Lab 04 baseline - agent v{agent.version}",
    data_source=data_source,
)
print(f"Evaluation: {eval_object.id}\nRun: {eval_run.id}")
run_reference = {"project_endpoint": project_endpoint, "agent_name": agent.name,
                 "agent_version": agent.version, "eval_id": eval_object.id, "run_id": eval_run.id}
(lab_dir / "evaluation-run.json").write_text(json.dumps(run_reference, indent=2), encoding="utf-8")
eval_run = wait_for_run(openai_client, eval_object.id, eval_run)
report_url = eval_run.report_url
if not report_url:
    raise RuntimeError(f"Completed run {eval_run.id} did not return a report URL. Inspect the project Evaluations page before continuing.")
print(f"Open this Foundry report: {report_url}")

## Part 6: Demonstrate the Results in Foundry

1. Open the **report URL printed above** in the same project. It is the service-provided link, not a hand-built portal URL.
2. From the agent's **Evaluation** tab choose **Automatic Evaluation**, not **Human Evaluation**. If the run is not listed there, open the project-level **Evaluations** page and find the run by name/ID. Portal layouts vary; the report link is authoritative.
3. Show all eight criteria and compare aggregate scores/pass rates. A failed quality check is an evaluation result, not a service failure.
4. Drill into a question: show the **actual agent response**, deterministic quality score (0-1), professional-tone score (1-5), pass/fail, and the tone judge's reason where returned. Code-based evaluators return a float only, not LLM reasoning.
5. Open the **evaluator catalog** to show the two reusable, versioned definitions. The custom-rubric banner is a separate creation experience; it is not required to run these code/prompt evaluators.
6. Run the export cells below. Keep **all 18 cases**, including blocked/error/skipped cases. The report explicitly sets `valid_for_comparison: false` when any generation or evaluator error/skip occurs; it must not be used as a CI/CD baseline. Missing scores are never counted as passes.

The existing agent is unchanged. This lab measures its current behavior; it does not add policy grounding or tools that the agent does not already have.

In [ ]:
import math
import pandas as pd

output_items = [item.model_dump(mode="json") for item in openai_client.evals.runs.output_items.list(
    eval_id=eval_object.id, run_id=eval_run.id)]
raw_results = {**run_reference, "report_url": report_url,
               "run": eval_run.model_dump(mode="json"), "output_items": output_items}
results_path = lab_dir / "evaluation-results.json"
results_path.write_text(json.dumps(raw_results, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Full cloud results (including reasons/errors): {results_path}")

def summarize_results(output_items, criteria, expected_count, run):
    if run.status != "completed" or len(output_items) != expected_count:
        raise ValueError("Run is incomplete or output-item count differs from the golden dataset. Inspect evaluation-results.json.")
    issues = []
    for criterion in run.per_testing_criteria_results or []:
        if getattr(criterion, "errored", 0) or getattr(criterion, "skipped", 0):
            issues.append(f"Evaluator {criterion.testing_criteria} has errored/skipped items.")
    names = [criterion["name"] for criterion in criteria]
    rows = []
    item_ids = set()
    for item in output_items:
        item_id = item["datasource_item_id"]
        if item_id in item_ids:
            raise ValueError(f"Duplicate output item {item_id}; inspect evaluation-results.json.")
        item_ids.add(item_id)
        source = item["datasource_item"]
        source = source.get("item", source)
        generation_error = item["sample"].get("error")
        response = "\n".join(message["content"] for message in item["sample"]["output"] if message.get("content"))
        generation_failed = bool(generation_error) or not response.strip() or item.get("status") in {"error", "errored", "skipped"}
        if generation_failed:
            issues.append(f"Item {item_id}: generation failed, was blocked/skipped, or returned no response. {generation_error or ''}")
        result_names = [result["name"] for result in item["results"]]
        if sorted(set(result_names)) != sorted(names):
            raise ValueError(f"Missing or unexpected evaluator results for item {item_id}.")
        duplicate_criteria = {name for name in result_names if result_names.count(name) > 1}
        if duplicate_criteria:
            issues.append(f"Item {item_id}: duplicate evaluator records {sorted(duplicate_criteria)}; all records retained, none used for comparison.")
        for result in item["results"]:
            score, passed = result.get("score"), result.get("passed")
            status = result.get("status", "completed")
            error = result.get("error") or (result.get("sample") or {}).get("error")
            evaluator_failed = bool(error) or status in {"error", "errored", "skipped"}
            if evaluator_failed:
                issues.append(f"Item {item_id}, {result['name']}: {status}. {error or result.get('reason') or ''}")
            elif not generation_failed and result['name'] not in duplicate_criteria and (isinstance(score, bool) or not isinstance(score, (int, float)) or not math.isfinite(score) or not isinstance(passed, bool)):
                raise ValueError(f"Invalid result for item {item_id}, evaluator {result['name']}; inspect evaluation-results.json.")
            rows.append({"item_id": item_id, "query": source["query"], "category": source["category"],
                         "response": response, "evaluator": result["name"], "score": score, "passed": passed,
                         "status": status, "generation_failed": generation_failed,
                         "usable_for_comparison": not generation_failed and not evaluator_failed and result['name'] not in duplicate_criteria,
                         "reason": result.get("reason") or result.get("reasoning"),
                         "error": error, "generation_error": generation_error})
    frame = pd.DataFrame(rows)
    summary = {}
    for name, group in frame.groupby("evaluator"):
        usable = group[group["usable_for_comparison"]]
        summary[name] = {"mean_score": float(usable["score"].mean()) if len(usable) else None,
                         "accuracy": float(usable["passed"].mean()) if len(usable) else None,
                         "defect_count": int(usable["passed"].eq(False).sum()), "sample_count": len(usable),
                         "total_count": int(group["item_id"].nunique()), "result_count": len(group),
                         "generation_error_count": int(group.loc[group["generation_failed"], "item_id"].nunique()),
                         "errored_count": int(group["status"].isin(["error", "errored"]).sum()),
                         "skipped_count": int(group["status"].eq("skipped").sum())}
    return frame, summary, issues

df_results, evaluator_scores, comparison_issues = summarize_results(output_items, testing_criteria, len(test_queries), eval_run)
valid_for_comparison = not comparison_issues
print(f"Valid for CI/CD comparison: {valid_for_comparison}. Averages use only successfully generated AND evaluated responses; see sample_count vs total_count.")
for issue in comparison_issues:
    print(f"WARNING: {issue}")
display(pd.DataFrame.from_dict(evaluator_scores, orient="index"))
display(df_results[["query", "response", "evaluator", "score", "passed", "status", "usable_for_comparison", "reason", "error", "generation_error"]])

In [ ]:
# Preserve all cases; invalid reports are diagnostic artifacts, NOT comparison baselines.
gate_names = {"Task Adherence": "task_adherence", "Coherence": "coherence", "Fluency": "fluency",
              "Violence": "violence", "Sexual Content": "sexual", "Hate/Unfairness": "hate_unfairness",
              "Response Quality Checker": "response_quality_checker", "Professional Tone": "professional_tone"}
baseline_report = {
    "evaluation_name": "Contoso Support Agent - Baseline",
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "source": "foundry_cloud_evaluation",
    "valid_for_comparison": valid_for_comparison,
    "comparison_issues": comparison_issues,
    **run_reference,
    "report_url": report_url,
    "results_file": results_path.name,
    "dataset_path": str(dataset_path.relative_to(repo_root)),
    "dataset_version": "v1",
    "evaluators": [tc["name"] for tc in testing_criteria],
    "testing_criteria": testing_criteria,
    "custom_evaluator_versions": {code_evaluator.name: code_evaluator.version, tone_evaluator.name: tone_evaluator.version},
    "scores": {gate_names[name]: score for name, score in evaluator_scores.items()},
    "results_summary": {
        "total_test_cases": len(output_items),
        "code_evaluator_pass_rate": evaluator_scores["Response Quality Checker"]["accuracy"],
        "code_evaluator_avg_score": evaluator_scores["Response Quality Checker"]["mean_score"],
    }
}

baseline_path = lab_dir / "baseline-report.json"
baseline_path.write_text(json.dumps(baseline_report, indent=2), encoding="utf-8")

print(f"Actual cloud report saved to {baseline_path}; valid_for_comparison={valid_for_comparison}")
if not valid_for_comparison:
    print("WARNING: Do NOT use this report for CI/CD comparisons. All cases and errors are retained for review; content filters were not bypassed.")
print(f"Portal report: {report_url}")

## ✅ Lab 04 Complete

### What You Created
- Reused the existing agent and pinned its version without modifying it
- One cloud evaluation with 6 built-in and 2 custom evaluators against actual agent responses
- Reusable, versioned **Response Quality Checker** and **Professional Tone** catalog entries
- Foundry report URL and actual per-item results including judge reasons and blocked/error/skipped cases

### Lab 7 Handoff
Load `baseline-report.json` and **require `valid_for_comparison` to be true** before using its `scores`. A report containing generation failures or evaluator errors/skips is diagnostic only, even when the cloud run status is completed. Keep all cases for review; do not bypass content filters, silently remove safety cases, or treat absent scores as passes. A candidate must use the **same dataset, evaluator definitions/versions, thresholds, and score scales**. The full run is in `evaluation-results.json`; `evaluation-run.json` records IDs even if polling times out. Lab 7's existing simulated gate examples remain illustrative: do not compare those synthetic values with these real scores. Groundedness and tool selection are not measured in this lab; do not invent scores for them.

### Key Takeaways
| Evaluator Type | Best For | Cost | Deterministic? |
|---------------|----------|------|---------------|
| Built-in | Standard quality/safety metrics | Judge model tokens | No |
| Code-based | Format, length, schema, rules | Free (compute only) | Yes |
| Prompt-based | Tone, domain quality, subjective | Judge model tokens | No |

### Next Steps
- **Lab 05**: Apply these evaluators to RAG optimization
- **Lab 06**: Add agent-specific trajectory evaluators